Cell 1: Setup & Load Data

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import ast
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data (same steps as EDA)
listings = pd.read_csv("../data/raw/listings.csv.gz")
listings['price_clean'] = listings['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
listings = listings[listings['price_clean'].notna() & (listings['price_clean'] > 0)]
listings = listings[listings['price_clean'] <= listings['price_clean'].quantile(0.99)].copy()
listings['log_price'] = np.log1p(listings['price_clean'])

print(f"✅ Loaded {listings.shape[0]} listings with {listings.shape[1]} columns")
print(f"💰 Target: price_clean (${listings['price_clean'].min():.0f} - ${listings['price_clean'].max():.0f})")
print(f"📊 Log price: log_price ({listings['log_price'].min():.2f} - {listings['log_price'].max():.2f})")

✅ Loaded 21200 listings with 81 columns
💰 Target: price_clean ($9 - $2261)
📊 Log price: log_price (2.30 - 7.72)


Cell 2: Select Base Features

In [12]:
# Select columns we'll keep
base_cols = [
    # Target
    'price_clean', 'log_price',
    
    # Location
    'neighbourhood_group_cleansed', 'neighbourhood_cleansed',
    'latitude', 'longitude',
    
    # Property
    'room_type', 'property_type', 'accommodates', 'bathrooms',
    'bedrooms', 'beds', 'amenities',
    
    # Booking
    'minimum_nights', 'maximum_nights', 'instant_bookable',
    
    # Availability
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    
    # Reviews
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value', 'reviews_per_month',
    
    # Host
    'host_is_superhost', 'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_listings_count', 'host_total_listings_count',
    'host_has_profile_pic', 'host_identity_verified',
]

df = listings[base_cols].copy()
print(f"✅ Selected {df.shape[1]} base features")
print(f"   Target columns: 2 (price_clean, log_price)")
print(f"   Feature columns: {df.shape[1]-2}")

✅ Selected 39 base features
   Target columns: 2 (price_clean, log_price)
   Feature columns: 37


Cell 3: Handle Missing Values

In [13]:
# Show missing values
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
print("📊 Missing Values (%):")
print(missing)

# Strategy:
# - Numeric columns: fill with median
# - Categorical columns: fill with mode

# Numeric columns to impute
numeric_fill_median = [
    'bathrooms', 'bedrooms', 'beds', 
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value', 'reviews_per_month',
    'host_listings_count', 'host_total_listings_count'
]

for col in numeric_fill_median:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)
        print(f"✅ {col}: filled NaN with median ({df[col].median():.1f})")

# Categorical columns
df['host_response_time'].fillna('unknown', inplace=True)

# Convert percentage strings to numeric
def pct_to_numeric(val):
    try:
        return float(val.replace('%', '')) / 100
    except:
        return np.nan

df['host_response_rate'] = df['host_response_rate'].apply(pct_to_numeric)
df['host_acceptance_rate'] = df['host_acceptance_rate'].apply(pct_to_numeric)

df['host_response_rate'].fillna(df['host_response_rate'].median(), inplace=True)
df['host_acceptance_rate'].fillna(df['host_acceptance_rate'].median(), inplace=True)

# Fill other categoricals
df['host_is_superhost'].fillna('f', inplace=True)
df['host_has_profile_pic'].fillna('f', inplace=True)
df['host_identity_verified'].fillna('f', inplace=True)

# Drop remaining rows with any NaN
before = df.shape[0]
df.dropna(inplace=True)
print(f"\n✅ Remaining NaN rows dropped: {before - df.shape[0]}")
print(f"   Final shape: {df.shape}")

📊 Missing Values (%):
review_scores_cleanliness      29.528302
review_scores_accuracy         29.528302
review_scores_rating           29.528302
review_scores_communication    29.528302
review_scores_location         29.528302
review_scores_value            29.528302
review_scores_checkin          29.528302
reviews_per_month              29.528302
host_acceptance_rate           21.207547
host_response_rate             20.726415
host_response_time             20.726415
host_has_profile_pic            6.250000
host_total_listings_count       6.250000
host_identity_verified          6.250000
host_listings_count             6.250000
host_is_superhost               1.740566
bedrooms                        0.551887
beds                            0.188679
bathrooms                       0.028302
dtype: float64
✅ bathrooms: filled NaN with median (1.0)
✅ bedrooms: filled NaN with median (1.0)
✅ beds: filled NaN with median (1.0)
✅ review_scores_rating: filled NaN with median (4.9)
✅ review_sc

Cell 4: Location Features

In [14]:
# Distance to Manhattan center (Times Square: 40.7580, -73.9855)
manhattan_center = (40.7580, -73.9855)

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km using Haversine formula"""
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['dist_to_center'] = haversine_distance(df['latitude'], df['longitude'], 
                                           manhattan_center[0], manhattan_center[1])

print(f"📍 Distance to Manhattan Center (Times Square):")
print(f"   Min: {df['dist_to_center'].min():.2f} km")
print(f"   Max: {df['dist_to_center'].max():.2f} km")
print(f"   Mean: {df['dist_to_center'].mean():.2f} km")

📍 Distance to Manhattan Center (Times Square):
   Min: 0.01 km
   Max: 35.14 km
   Mean: 7.92 km


Cell 5: Amenities Feature Engineering

In [15]:
# Parse amenities list
def parse_amenities(amenity_str):
    try:
        return ast.literal_eval(amenity_str)
    except:
        return []

def count_amenities(amenity_str):
    return len(parse_amenities(amenity_str))

# Basic count
df['amenities_count'] = df['amenities'].apply(count_amenities)

# Key amenities indicators
key_amenities = [
    'Wifi', 'Kitchen', 'Air conditioning', 'Heating', 
    'Washer', 'Dryer', 'TV', 'Pool', 'Gym', 'Elevator',
    'Free parking', 'Pet friendly', 'Smoke detector', 'Fire extinguisher'
]

for amenity in key_amenities:
    col_name = 'has_' + amenity.lower().replace(' ', '_')
    df[col_name] = df['amenities'].str.contains(amenity, case=False, na=False).astype(int)

# Count how many premium amenities each listing has
premium_amenities = ['Pool', 'Gym', 'Elevator', 'Free parking']
df['premium_count'] = df[[f'has_{a.lower().replace(" ", "_")}' for a in premium_amenities]].sum(axis=1)

print(f"🔧 Amenities Features Created:")
print(f"   • amenities_count: {df['amenities_count'].mean():.0f} avg amenities")
print(f"   • Premium amenities count: {df['premium_count'].mean():.2f} avg")
print(f"   • {len(key_amenities)} individual amenity indicators")

# Drop original amenities column
df.drop('amenities', axis=1, inplace=True)

🔧 Amenities Features Created:
   • amenities_count: 34 avg amenities
   • Premium amenities count: 0.45 avg
   • 14 individual amenity indicators


Cell 6: Encode Categorical Variables

In [16]:
from sklearn.preprocessing import LabelEncoder

print("="*60)
print("🔄 ENCODING CATEGORICAL VARIABLES")
print("="*60)

# Binary encoding for f/t columns
binary_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
for col in binary_cols:
    df[col] = (df[col] == 't').astype(int)
    print(f"✅ {col}: t/f → 1/0")

# Label encode neighbourhood (keep original name for reference)
le_neigh = LabelEncoder()
df['neighbourhood_cleansed_encoded'] = le_neigh.fit_transform(df['neighbourhood_cleansed'])
print(f"✅ neighbourhood_cleansed: {df['neighbourhood_cleansed'].nunique()} unique → encoded")

# One-hot encode room_type, neighbourhood_group, property_type, host_response_time
categorical_to_encode = ['room_type', 'neighbourhood_group_cleansed', 'property_type', 'host_response_time']
df = pd.get_dummies(df, columns=categorical_to_encode, drop_first=True, sparse=False)

print(f"✅ One-hot encoding done!")
print(f"   Shape after encoding: {df.shape}")

# Drop original categorical columns that are now encoded
cols_to_drop = ['neighbourhood_cleansed']  # We have encoded version
if 'neighbourhood_cleansed' in df.columns:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"✅ Dropped original 'neighbourhood_cleansed' column (encoded version kept)")

# Verify no string columns remain
string_cols = df.select_dtypes(include=['object']).columns.tolist()
if string_cols:
    print(f"\n⚠️  Warning: {len(string_cols)} string columns remain:")
    for col in string_cols:
        unique_vals = df[col].nunique()
        sample_vals = df[col].dropna().unique()[:3]
        print(f"   • {col}: {unique_vals} unique values | Sample: {sample_vals}")
else:
    print(f"✅ No string columns remaining - all numeric!")

print(f"\n📊 Final columns: {df.shape[1]}")

🔄 ENCODING CATEGORICAL VARIABLES
✅ host_is_superhost: t/f → 1/0
✅ host_has_profile_pic: t/f → 1/0
✅ host_identity_verified: t/f → 1/0
✅ instant_bookable: t/f → 1/0
✅ neighbourhood_cleansed: 206 unique → encoded
✅ One-hot encoding done!
   Shape after encoding: (11923, 112)
✅ Dropped original 'neighbourhood_cleansed' column (encoded version kept)
✅ No string columns remaining - all numeric!

📊 Final columns: 111


Cell 7: Feature Scaling

In [17]:
# Separate features and target
feature_cols = [col for col in df.columns if col not in ['price_clean', 'log_price']]
X = df[feature_cols]
y = df['log_price']  # We'll predict log price
y_original = df['price_clean']

print(f"📊 Final Dataset:")
print(f"   Features (X): {X.shape}")
print(f"   Target (y - log_price): {y.shape}")
print(f"   Original price: {y_original.shape}")
print(f"\n📋 Feature columns:")
for i, col in enumerate(X.columns):
    print(f"   {i+1}. {col}")

📊 Final Dataset:
   Features (X): (11923, 109)
   Target (y - log_price): (11923,)
   Original price: (11923,)

📋 Feature columns:
   1. latitude
   2. longitude
   3. accommodates
   4. bathrooms
   5. bedrooms
   6. beds
   7. minimum_nights
   8. maximum_nights
   9. instant_bookable
   10. availability_30
   11. availability_60
   12. availability_90
   13. availability_365
   14. number_of_reviews
   15. number_of_reviews_ltm
   16. number_of_reviews_l30d
   17. review_scores_rating
   18. review_scores_accuracy
   19. review_scores_cleanliness
   20. review_scores_checkin
   21. review_scores_communication
   22. review_scores_location
   23. review_scores_value
   24. reviews_per_month
   25. host_is_superhost
   26. host_response_rate
   27. host_acceptance_rate
   28. host_listings_count
   29. host_total_listings_count
   30. host_has_profile_pic
   31. host_identity_verified
   32. dist_to_center
   33. amenities_count
   34. has_wifi
   35. has_kitchen
   36. has_air_condit

Cell 8: Train-Test Split

In [18]:
# Split into train (70%), validation (15%), test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, random_state=42
)

print(f"📊 Data Splits:")
print(f"   X_train: {X_train.shape}")
print(f"   X_val:   {X_val.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"\n   y_train: {y_train.shape}")
print(f"   y_val:   {y_val.shape}")
print(f"   y_test:  {y_test.shape}")

# Check target distribution
print(f"\n📈 Log Price Distribution:")
print(f"   Train: {y_train.mean():.2f} ± {y_train.std():.2f}")
print(f"   Val:   {y_val.mean():.2f} ± {y_val.std():.2f}")
print(f"   Test:  {y_test.mean():.2f} ± {y_test.std():.2f}")

📊 Data Splits:
   X_train: (8345, 109)
   X_val:   (1789, 109)
   X_test:  (1789, 109)

   y_train: (8345,)
   y_val:   (1789,)
   y_test:  (1789,)

📈 Log Price Distribution:
   Train: 5.01 ± 0.71
   Val:   5.01 ± 0.71
   Test:  5.01 ± 0.72


Cell 9: Save Processed Data

In [19]:
# Save all processed data
processed_dir = "../data/processed"
import os
os.makedirs(processed_dir, exist_ok=True)

# Save data splits
X_train.to_csv(f"{processed_dir}/X_train.csv", index=False)
X_val.to_csv(f"{processed_dir}/X_val.csv", index=False)
X_test.to_csv(f"{processed_dir}/X_test.csv", index=False)

y_train.to_csv(f"{processed_dir}/y_train.csv", index=False, header=True)
y_val.to_csv(f"{processed_dir}/y_val.csv", index=False, header=True)
y_test.to_csv(f"{processed_dir}/y_test.csv", index=False, header=True)

# Save feature list for later use
feature_cols_df = pd.DataFrame({'feature': X.columns.tolist()})
feature_cols_df.to_csv(f"{processed_dir}/feature_list.csv", index=False)

# Also save the entire processed dataframe (for reference)
df.to_csv(f"{processed_dir}/processed_listings.csv", index=False)

print(f"✅ All processed data saved to {processed_dir}/")
print(f"   • X_train.csv, X_val.csv, X_test.csv")
print(f"   • y_train.csv, y_val.csv, y_test.csv")
print(f"   • feature_list.csv")
print(f"   • processed_listings.csv")

✅ All processed data saved to ../data/processed/
   • X_train.csv, X_val.csv, X_test.csv
   • y_train.csv, y_val.csv, y_test.csv
   • feature_list.csv
   • processed_listings.csv


Cell 10: Phase 3 Summary

In [20]:
print("="*65)
print("🔧 PHASE 3 COMPLETE - FEATURE ENGINEERING SUMMARY")
print("="*65)

print(f"\n📦 Data Overview:")
print(f"   • Total samples: {df.shape[0]:,}")
print(f"   • Total features: {X.shape[1]}")
print(f"   • Target: log_price (will convert back to $ after prediction)")

print(f"\n🔑 Feature Groups:")
print(f"   📍 Location: distance to center, neighbourhood encoded, borough dummies")
print(f"   🏠 Property: room type, property type, accommodates, bedrooms, bathrooms, beds")
print(f"   🔧 Amenities: count + {len(key_amenities)} individual indicators")
print(f"   ⭐ Host: superhost status, response rate, acceptance rate, verifications")
print(f"   📝 Reviews: scores (rating, accuracy, cleanliness, etc.), review counts")
print(f"   📅 Booking: minimum/maximum nights, instant bookable, availability")

print(f"\n📊 Train/Val/Test Split:")
print(f"   • Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/df.shape[0]*100:.0f}%)")
print(f"   • Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/df.shape[0]*100:.0f}%)")
print(f"   • Test: {X_test.shape[0]:,} samples ({X_test.shape[0]/df.shape[0]*100:.0f}%)")

print(f"\n📁 Files Saved:")
print(f"   ✅ data/processed/X_train.csv")
print(f"   ✅ data/processed/X_val.csv")
print(f"   ✅ data/processed/X_test.csv")
print(f"   ✅ data/processed/y_train.csv")
print(f"   ✅ data/processed/y_val.csv")
print(f"   ✅ data/processed/y_test.csv")
print(f"   ✅ data/processed/feature_list.csv")
print(f"   ✅ data/processed/processed_listings.csv")

print(f"\n{'='*65}")
print(f"🚀 READY FOR PHASE 4: MODEL BUILDING & TRAINING!")
print(f"{'='*65}")

🔧 PHASE 3 COMPLETE - FEATURE ENGINEERING SUMMARY

📦 Data Overview:
   • Total samples: 11,923
   • Total features: 109
   • Target: log_price (will convert back to $ after prediction)

🔑 Feature Groups:
   📍 Location: distance to center, neighbourhood encoded, borough dummies
   🏠 Property: room type, property type, accommodates, bedrooms, bathrooms, beds
   🔧 Amenities: count + 14 individual indicators
   ⭐ Host: superhost status, response rate, acceptance rate, verifications
   📝 Reviews: scores (rating, accuracy, cleanliness, etc.), review counts
   📅 Booking: minimum/maximum nights, instant bookable, availability

📊 Train/Val/Test Split:
   • Train: 8,345 samples (70%)
   • Validation: 1,789 samples (15%)
   • Test: 1,789 samples (15%)

📁 Files Saved:
   ✅ data/processed/X_train.csv
   ✅ data/processed/X_val.csv
   ✅ data/processed/X_test.csv
   ✅ data/processed/y_train.csv
   ✅ data/processed/y_val.csv
   ✅ data/processed/y_test.csv
   ✅ data/processed/feature_list.csv
   ✅ data/p